# Cálculo da Taxa por 100 Mil Habitantes — Todas as Variáveis

Calcula a taxa ajustada pela população, por município e por ano, para
as quatro variáveis de evento do trabalho: **Casos**, **Mortalidade**,
**Mamografia** e **Hospitais Habilitados**.

$$\text{Taxa} = \frac{\text{Quantidade do evento}}{\text{População}} \times 100.000$$



## Imports

In [1]:
import pandas as pd
from pathlib import Path

from utilitarios_populacional import (
    COLUNA_MUNICIPIO,
    COLUNA_CODIGO_IBGE,
    COLUNA_ANO,
    COLUNA_TIPO,
    COLUNA_QUANTIDADE,
    COLUNA_POPULACAO,
    COLUNA_TAXA,
    PREFIXO_COLUNA_TOTAL_ANO,
    MULTIPLICADOR_TAXA,
    registrar_log,
    normalizar_codigo_ibge,
)

## Configuração Geral

In [ ]:
CAMINHO_POPULACAO = Path("/home/anna/tcc/Mineração de Dados - Avaliação e Apresentação/Análise Populacional/populacao_interpolacao_extrapolacao_geometrica.csv")

PASTA_SAIDA = Path("/home/anna/tcc/Mineração de Dados - Avaliação e Apresentação/Análise Populacional")


LISTA_VARIAVEIS = [
    {
        "NOME": "Casos",
        "CAMINHO_ENTRADA": Path("/home/anna/tcc/Seleção e Transformação/Casos/incidencia _ano_diagnostico.csv"),
        "ANO_INICIAL": 1985,
        "ANO_FINAL": 2023,
        "TEM_MULTIPLAS_LINHAS_POR_MUNICIPIO": False,
        "NOME_ARQUIVO_SAIDA": "casos_taxa_por_100_mil.csv",
    },
    {
        "NOME": "Mortalidade",
        "CAMINHO_ENTRADA": Path("/home/anna/tcc/Seleção e Transformação/Mortalidade/tabela_mortalidade_faixa_etaria.csv"),
        "ANO_INICIAL": 1979,
        "ANO_FINAL": 2024,
        "TEM_MULTIPLAS_LINHAS_POR_MUNICIPIO": False,
        "NOME_ARQUIVO_SAIDA": "mortalidade_taxa_por_100_mil.csv",
    },
    {
        "NOME": "Mamografia",
        "CAMINHO_ENTRADA": Path("/home/anna/tcc/Seleção e Transformação/Mamografias/tabela_mamografia.csv"),
        "ANO_INICIAL": 2013,
        "ANO_FINAL": 2025,
        "TEM_MULTIPLAS_LINHAS_POR_MUNICIPIO": False,
        "NOME_ARQUIVO_SAIDA": "mamografia_taxa_por_100_mil.csv",
    },
    {
        "NOME": "Hospitais",
        "CAMINHO_ENTRADA": Path("/home/anna/tcc/Seleção e Transformação/Hospitais/tabela_hospitais_habilitados.csv"),
        "ANO_INICIAL": 2023,
        "ANO_FINAL": 2023,
        "TEM_MULTIPLAS_LINHAS_POR_MUNICIPIO": True,
        "NOME_ARQUIVO_SAIDA": "hospitais_taxa_por_100_mil.csv",
    },
]

LISTA_VARIAVEIS_CATEGORICAS = [
    {
        "NOME": "Casos por Escolaridade",
        "CAMINHO_ENTRADA": Path("/home/anna/tcc/Seleção e Transformação/Casos/incidencia _escolaridade.csv"),
        "ANO_INICIAL": 1998,
        "ANO_FINAL": 2023,
        "NOME_ARQUIVO_SAIDA": "casos_escolaridade_taxa_por_100_mil.csv",
    },
    {
        "NOME": "Casos por Raça e Cor",
        "CAMINHO_ENTRADA": Path("/home/anna/tcc/Seleção e Transformação/Casos/incidencia _raca_cor.csv"),
        "ANO_INICIAL": 1998,
        "ANO_FINAL": 2023,
        "NOME_ARQUIVO_SAIDA": "casos_raca_cor_taxa_por_100_mil.csv",
    },
    {
        "NOME": "Casos por Faixa Etária",
        "CAMINHO_ENTRADA": Path("/home/anna/tcc/Seleção e Transformação/Casos/incidencia _faixa_etaria.csv"),
        "ANO_INICIAL": 1985,
        "ANO_FINAL": 2023,
        "NOME_ARQUIVO_SAIDA": "casos_faixa_etaria_taxa_por_100_mil.csv",
    },
]

CAMINHO_HOSPITAIS = Path("/home/anna/tcc/Seleção e Transformação/Hospitais/tabela_hospitais_habilitados.csv")
ANO_REFERENCIA_HOSPITAIS = 2023
NOME_ARQUIVO_SAIDA_HOSPITAIS_POR_TIPO = "hospitais_por_tipo_taxa_por_100_mil.csv"

## Funções Genéricas


In [ ]:
def transformar_populacao_em_formato_longo(tabela_populacao):
    # Lista para guardar os nomes das colunas de ano (TOTAL_1970, TOTAL_1971, ...)
    colunas_de_ano = []
    for coluna in tabela_populacao.columns:
        if coluna.startswith(PREFIXO_COLUNA_TOTAL_ANO):  # filtra só colunas que começam com "TOTAL_"
            colunas_de_ano.append(coluna)  # adiciona à lista de colunas de ano

    # Converte a tabela de larga (uma coluna por ano) para longa (uma linha por município x ano)
    tabela_longa = tabela_populacao.melt(
        id_vars=[COLUNA_MUNICIPIO, COLUNA_CODIGO_IBGE],  # colunas que se repetem, não viram linha
        value_vars=colunas_de_ano,  # colunas que serão "empilhadas" em linhas
        var_name=COLUNA_ANO,  # nome da coluna que vai guardar o nome original (ex.: "TOTAL_1970")
        value_name=COLUNA_POPULACAO,  # nome da coluna que vai guardar o valor da população
    )

    # Lista para guardar os anos já convertidos de texto ("TOTAL_1970") para número (1970)
    lista_anos_convertidos = []
    for valor_da_coluna_ano in tabela_longa[COLUNA_ANO]:
        ano_em_texto = valor_da_coluna_ano.replace(PREFIXO_COLUNA_TOTAL_ANO, "")  # remove o prefixo "TOTAL_"
        lista_anos_convertidos.append(int(ano_em_texto))  # converte o texto restante para inteiro

    tabela_longa[COLUNA_ANO] = lista_anos_convertidos  # substitui a coluna de ano pelos valores já convertidos

    return tabela_longa[[COLUNA_CODIGO_IBGE, COLUNA_ANO, COLUNA_POPULACAO]]  # mantém só as colunas necessárias


def somar_linhas_duplicadas_por_municipio(tabela, ano_inicial, ano_final):
    # Lista para guardar as colunas de ano que de fato existem na tabela, dentro do período pedido
    colunas_totais_do_periodo = []
    for ano in range(ano_inicial, ano_final + 1):
        nome_da_coluna = f"{PREFIXO_COLUNA_TOTAL_ANO}{ano}"  # monta o nome esperado, ex.: "TOTAL_2023"
        if nome_da_coluna in tabela.columns:  # só considera se a coluna realmente existir
            colunas_totais_do_periodo.append(nome_da_coluna)

    # Agrupa por município (nome + código IBGE) e soma as colunas de ano,
    # colapsando as múltiplas linhas de um mesmo município em uma só
    tabela_agrupada = tabela.groupby([COLUNA_MUNICIPIO, COLUNA_CODIGO_IBGE], as_index=False)[colunas_totais_do_periodo].sum()

    return tabela_agrupada


def transformar_variavel_em_formato_longo(tabela, ano_inicial, ano_final):
    # Lista para guardar as colunas de ano que existem na tabela, dentro do período pedido
    colunas_de_ano = []
    for ano in range(ano_inicial, ano_final + 1):
        nome_da_coluna = f"{PREFIXO_COLUNA_TOTAL_ANO}{ano}"  # monta o nome esperado, ex.: "TOTAL_1985"
        if nome_da_coluna in tabela.columns:  # só considera se a coluna existir na tabela
            colunas_de_ano.append(nome_da_coluna)

    # Converte a tabela de larga (uma coluna por ano) para longa (uma linha por município x ano)
    tabela_longa = tabela.melt(
        id_vars=[COLUNA_MUNICIPIO, COLUNA_CODIGO_IBGE],  # colunas que se repetem, não viram linha
        value_vars=colunas_de_ano,  # colunas que serão "empilhadas" em linhas
        var_name=COLUNA_ANO,  # nome da coluna que vai guardar o nome original da coluna de ano
        value_name=COLUNA_QUANTIDADE,  # nome da coluna que vai guardar a quantidade do evento
    )

    # Lista para guardar os anos já convertidos de texto para número
    lista_anos_convertidos = []
    for valor_da_coluna_ano in tabela_longa[COLUNA_ANO]:
        ano_em_texto = valor_da_coluna_ano.replace(PREFIXO_COLUNA_TOTAL_ANO, "")  # remove o prefixo "TOTAL_"
        lista_anos_convertidos.append(int(ano_em_texto))  # converte para inteiro

    tabela_longa[COLUNA_ANO] = lista_anos_convertidos  # substitui a coluna de ano pelos valores convertidos

    return tabela_longa[[COLUNA_MUNICIPIO, COLUNA_CODIGO_IBGE, COLUNA_ANO, COLUNA_QUANTIDADE]]  # colunas finais


def transformar_variavel_categorica_em_formato_longo(tabela, ano_inicial, ano_final):
    # Lista das colunas que seguem o padrão "<ano>_<categoria>" e serão transformadas
    colunas_categoricas = []
    # Dicionário para lembrar, para cada coluna original, qual ano e categoria ela representa
    mapa_coluna_para_ano_categoria = {}

    for coluna in tabela.columns:
        if coluna in (COLUNA_MUNICIPIO, COLUNA_CODIGO_IBGE):
            continue  # ignora as colunas de identificação do município
        if coluna.startswith(PREFIXO_COLUNA_TOTAL_ANO):
            continue  # TOTAL_<ano> não tem categoria — fica de fora

        ano_texto, separador, categoria = coluna.partition("_")  # separa "2010_BRANCA" em "2010", "_", "BRANCA"
        if not separador:
            continue  # não segue o padrão "<ano>_<categoria>"

        try:
            ano = int(ano_texto)  # tenta converter a primeira parte para um ano numérico
        except ValueError:
            continue  # primeira parte não é um ano válido, ignora a coluna

        if ano_inicial <= ano <= ano_final:  # só mantém colunas dentro do período pedido
            colunas_categoricas.append(coluna)
            mapa_coluna_para_ano_categoria[coluna] = (ano, categoria)  # guarda o ano e a categoria da coluna

    # Converte a tabela de larga (uma coluna por ano+categoria) para longa (uma linha por combinação)
    tabela_longa = tabela.melt(
        id_vars=[COLUNA_MUNICIPIO, COLUNA_CODIGO_IBGE],  # colunas que se repetem, não viram linha
        value_vars=colunas_categoricas,  # colunas que serão "empilhadas" em linhas
        var_name="COLUNA_ORIGINAL",  # guarda temporariamente o nome original da coluna (ex.: "2010_BRANCA")
        value_name=COLUNA_QUANTIDADE,  # guarda o valor da quantidade
    )

    # Usa o mapa criado antes para recuperar o ano de cada linha, a partir do nome original da coluna
    tabela_longa[COLUNA_ANO] = tabela_longa["COLUNA_ORIGINAL"].map(
        lambda c: mapa_coluna_para_ano_categoria[c][0]
    )
    # Usa o mesmo mapa para recuperar a categoria de cada linha
    tabela_longa["CATEGORIA"] = tabela_longa["COLUNA_ORIGINAL"].map(
        lambda c: mapa_coluna_para_ano_categoria[c][1]
    )
    tabela_longa = tabela_longa.drop(columns=["COLUNA_ORIGINAL"])  # remove a coluna auxiliar, não é mais necessária

    return tabela_longa


def transformar_taxas_categoricas_em_formato_largo(tabela_com_taxa):
    tabela_larga = tabela_com_taxa.copy()  # copia para não alterar a tabela original recebida
    # Recria o nome da coluna original (ex.: "2010_BRANCA"), juntando ano e categoria de volta
    tabela_larga["ANO_CATEGORIA"] = (
        tabela_larga[COLUNA_ANO].astype(str) + "_" + tabela_larga["CATEGORIA"].astype(str)
    )

    # Pivota a tabela: cada combinação ano_categoria vira uma coluna, com a taxa como valor
    tabela_larga = tabela_larga.pivot(
        index=[COLUNA_MUNICIPIO, COLUNA_CODIGO_IBGE],  # cada linha do resultado é um município
        columns="ANO_CATEGORIA",  # cada valor único dessa coluna vira uma coluna nova
        values=COLUNA_TAXA,  # os valores preenchidos são as taxas calculadas
    ).reset_index()  # traz MUNICÍPIOS e CÓDIGO IBGE de volta como colunas normais

    tabela_larga.columns.name = None  # remove o nome do índice de colunas criado pelo pivot (só estética)
    return tabela_larga


def preparar_hospitais_por_tipo_em_formato_longo(tabela, ano_referencia):
    coluna_quantidade_no_ano = f"{PREFIXO_COLUNA_TOTAL_ANO}{ano_referencia}"  # monta o nome da coluna do ano pedido

    # Seleciona só as colunas necessárias, mantendo a coluna TIPO (sem somar os tipos entre si)
    tabela_longa = tabela[[COLUNA_MUNICIPIO, COLUNA_CODIGO_IBGE, COLUNA_TIPO, coluna_quantidade_no_ano]].copy()
    tabela_longa = tabela_longa.rename(columns={coluna_quantidade_no_ano: COLUNA_QUANTIDADE})  # padroniza o nome da coluna de valor
    tabela_longa[COLUNA_ANO] = ano_referencia  # adiciona a coluna de ano, fixa no ano de referência

    return tabela_longa[[COLUNA_MUNICIPIO, COLUNA_CODIGO_IBGE, COLUNA_ANO, COLUNA_TIPO, COLUNA_QUANTIDADE]]


def cruzar_com_populacao_e_calcular_taxa(tabela_evento_longa, tabela_populacao_longa, nome_da_variavel):
    # Junta a tabela do evento com a tabela de população, casando por Código IBGE e Ano
    tabela_cruzada = pd.merge(
        tabela_evento_longa,
        tabela_populacao_longa,
        on=[COLUNA_CODIGO_IBGE, COLUNA_ANO],
        how="left",  # mantém todas as linhas do evento, mesmo sem população correspondente
    )

    # Conta quantas linhas ficaram sem população (não encontraram par no merge)
    quantidade_de_linhas_sem_populacao = tabela_cruzada[COLUNA_POPULACAO].isna().sum()

    if quantidade_de_linhas_sem_populacao > 0:
        registrar_log(  # avisa no console quantas linhas ficaram sem população correspondente
            f"ATENÇÃO ({nome_da_variavel}): {quantidade_de_linhas_sem_populacao} linha(s) "
            f"não encontraram população correspondente (município ou ano fora do "
            f"intervalo disponível). A taxa dessas linhas ficará como NaN."
        )
    else:
        registrar_log(f"Cruzamento com população ({nome_da_variavel}) concluído sem inconsistências.")

    populacao_valida = tabela_cruzada[COLUNA_POPULACAO].replace(0, pd.NA)  # troca população 0 por NA, evita divisão por zero

    # Calcula a taxa por 100 mil habitantes: quantidade dividida pela população, multiplicada pelo fator
    tabela_cruzada[COLUNA_TAXA] = (tabela_cruzada[COLUNA_QUANTIDADE] / populacao_valida) * MULTIPLICADOR_TAXA

    return tabela_cruzada


def salvar_resultado(tabela, caminho_de_saida):
    caminho_de_saida.parent.mkdir(parents=True, exist_ok=True)  # cria a pasta de saída, se ainda não existir
    tabela.to_csv(caminho_de_saida, index=False)  # salva a tabela em CSV, sem escrever o índice
    registrar_log(f"Arquivo salvo em: {caminho_de_saida}")  # avisa no console onde o arquivo foi salvo

## Carregamento da População (uma única vez)

In [4]:
tabela_populacao_bruta = pd.read_csv(CAMINHO_POPULACAO)
tabela_populacao_bruta[COLUNA_CODIGO_IBGE] = normalizar_codigo_ibge(tabela_populacao_bruta[COLUNA_CODIGO_IBGE])
tabela_populacao_longa = transformar_populacao_em_formato_longo(tabela_populacao_bruta)

registrar_log(f"População carregada e transformada: {tabela_populacao_longa.shape[0]} linhas (município x ano).")
tabela_populacao_longa.head()

[LOG] População carregada e transformada: 47768 linhas (município x ano).


,CÓDIGO IBGE,ANO,POPULACAO
0,3100104,1970,4848.0
1,3100203,1970,10206.0
2,3100302,1970,9779.0
3,3100401,1970,1699.0
4,3100807,1970,1312.0


## Processamento de Todas as Variáveis


In [5]:
dicionario_resultados_por_variavel = {}

for configuracao_da_variavel in LISTA_VARIAVEIS:

    nome_da_variavel = configuracao_da_variavel["NOME"]
    registrar_log(f"===== INÍCIO: {nome_da_variavel.upper()} =====")

    tabela_bruta = pd.read_csv(configuracao_da_variavel["CAMINHO_ENTRADA"])
    tabela_bruta[COLUNA_CODIGO_IBGE] = normalizar_codigo_ibge(tabela_bruta[COLUNA_CODIGO_IBGE])

    if configuracao_da_variavel.get("TEM_MULTIPLAS_LINHAS_POR_MUNICIPIO", False):
        tabela_bruta = somar_linhas_duplicadas_por_municipio(
            tabela_bruta,
            configuracao_da_variavel["ANO_INICIAL"],
            configuracao_da_variavel["ANO_FINAL"],
        )

    tabela_evento_longa = transformar_variavel_em_formato_longo(
        tabela_bruta,
        configuracao_da_variavel["ANO_INICIAL"],
        configuracao_da_variavel["ANO_FINAL"],
    )

    tabela_com_taxa = cruzar_com_populacao_e_calcular_taxa(
        tabela_evento_longa, tabela_populacao_longa, nome_da_variavel
    )

    caminho_de_saida = PASTA_SAIDA / configuracao_da_variavel["NOME_ARQUIVO_SAIDA"]
    salvar_resultado(tabela_com_taxa, caminho_de_saida)

    dicionario_resultados_por_variavel[nome_da_variavel] = tabela_com_taxa

    registrar_log(f"===== FIM: {nome_da_variavel.upper()} — {tabela_com_taxa.shape[0]} linhas =====\n")


for configuracao_da_variavel in LISTA_VARIAVEIS_CATEGORICAS:

    nome_da_variavel = configuracao_da_variavel["NOME"]
    registrar_log(f"===== INÍCIO: {nome_da_variavel.upper()} =====")

    tabela_bruta = pd.read_csv(configuracao_da_variavel["CAMINHO_ENTRADA"])
    tabela_bruta[COLUNA_CODIGO_IBGE] = normalizar_codigo_ibge(tabela_bruta[COLUNA_CODIGO_IBGE])

    tabela_evento_longa = transformar_variavel_categorica_em_formato_longo(
        tabela_bruta,
        configuracao_da_variavel["ANO_INICIAL"],
        configuracao_da_variavel["ANO_FINAL"],
    )

    tabela_com_taxa = cruzar_com_populacao_e_calcular_taxa(
        tabela_evento_longa, tabela_populacao_longa, nome_da_variavel
    )

    tabela_com_taxa = transformar_taxas_categoricas_em_formato_largo(tabela_com_taxa)

    caminho_de_saida = PASTA_SAIDA / configuracao_da_variavel["NOME_ARQUIVO_SAIDA"]
    salvar_resultado(tabela_com_taxa, caminho_de_saida)

    dicionario_resultados_por_variavel[nome_da_variavel] = tabela_com_taxa

    registrar_log(f"===== FIM: {nome_da_variavel.upper()} — {tabela_com_taxa.shape[0]} linhas =====\n")

[LOG] ===== INÍCIO: CASOS =====
[LOG] ATENÇÃO (Casos): 780 linha(s) não encontraram população correspondente (município ou ano fora do intervalo disponível). A taxa dessas linhas ficará como NaN.
[LOG] Arquivo salvo em: /home/anna/tcc/Mineração de Dados - Avaliação e Apresentação/Análise Populacional/casos_taxa_por_100_mil.csv
[LOG] ===== FIM: CASOS — 33267 linhas =====

[LOG] ===== INÍCIO: MORTALIDADE =====
[LOG] ATENÇÃO (Mortalidade): 1387 linha(s) não encontraram população correspondente (município ou ano fora do intervalo disponível). A taxa dessas linhas ficará como NaN.


[LOG] Arquivo salvo em: /home/anna/tcc/Mineração de Dados - Avaliação e Apresentação/Análise Populacional/mortalidade_taxa_por_100_mil.csv
[LOG] ===== FIM: MORTALIDADE — 38686 linhas =====

[LOG] ===== INÍCIO: MAMOGRAFIA =====
[LOG] ATENÇÃO (Mamografia): 78 linha(s) não encontraram população correspondente (município ou ano fora do intervalo disponível). A taxa dessas linhas ficará como NaN.
[LOG] Arquivo salvo em: /home/anna/tcc/Mineração de Dados - Avaliação e Apresentação/Análise Populacional/mamografia_taxa_por_100_mil.csv
[LOG] ===== FIM: MAMOGRAFIA — 11076 linhas =====

[LOG] ===== INÍCIO: HOSPITAIS =====
[LOG] Cruzamento com população (Hospitais) concluído sem inconsistências.
[LOG] Arquivo salvo em: /home/anna/tcc/Mineração de Dados - Avaliação e Apresentação/Análise Populacional/hospitais_taxa_por_100_mil.csv
[LOG] ===== FIM: HOSPITAIS — 26 linhas =====

[LOG] ===== INÍCIO: CASOS POR ESCOLARIDADE =====
[LOG] Cruzamento com população (Casos por Escolaridade) concluído sem incon

[LOG] Arquivo salvo em: /home/anna/tcc/Mineração de Dados - Avaliação e Apresentação/Análise Populacional/casos_escolaridade_taxa_por_100_mil.csv
[LOG] ===== FIM: CASOS POR ESCOLARIDADE — 853 linhas =====

[LOG] ===== INÍCIO: CASOS POR RAÇA E COR =====
[LOG] Cruzamento com população (Casos por Raça e Cor) concluído sem inconsistências.


[LOG] Arquivo salvo em: /home/anna/tcc/Mineração de Dados - Avaliação e Apresentação/Análise Populacional/casos_raca_cor_taxa_por_100_mil.csv
[LOG] ===== FIM: CASOS POR RAÇA E COR — 853 linhas =====

[LOG] ===== INÍCIO: CASOS POR FAIXA ETÁRIA =====
[LOG] ATENÇÃO (Casos por Faixa Etária): 5200 linha(s) não encontraram população correspondente (município ou ano fora do intervalo disponível). A taxa dessas linhas ficará como NaN.


[LOG] Arquivo salvo em: /home/anna/tcc/Mineração de Dados - Avaliação e Apresentação/Análise Populacional/casos_faixa_etaria_taxa_por_100_mil.csv
[LOG] ===== FIM: CASOS POR FAIXA ETÁRIA — 853 linhas =====



##  Hospitais por Tipo (SEM somar os 3 tipos)


In [6]:
tabela_hospitais_bruta = pd.read_csv(CAMINHO_HOSPITAIS)
tabela_hospitais_bruta[COLUNA_CODIGO_IBGE] = normalizar_codigo_ibge(tabela_hospitais_bruta[COLUNA_CODIGO_IBGE])

tabela_hospitais_por_tipo_longa = preparar_hospitais_por_tipo_em_formato_longo(
    tabela_hospitais_bruta, ANO_REFERENCIA_HOSPITAIS
)

tabela_hospitais_por_tipo_com_taxa = cruzar_com_populacao_e_calcular_taxa(
    tabela_hospitais_por_tipo_longa, tabela_populacao_longa, "Hospitais por Tipo"
)

caminho_de_saida_hospitais_por_tipo = PASTA_SAIDA / NOME_ARQUIVO_SAIDA_HOSPITAIS_POR_TIPO
salvar_resultado(tabela_hospitais_por_tipo_com_taxa, caminho_de_saida_hospitais_por_tipo)

dicionario_resultados_por_variavel["Hospitais por Tipo"] = tabela_hospitais_por_tipo_com_taxa

registrar_log(f"===== FIM: HOSPITAIS POR TIPO — {tabela_hospitais_por_tipo_com_taxa.shape[0]} linhas =====")

[LOG] Cruzamento com população (Hospitais por Tipo) concluído sem inconsistências.
[LOG] Arquivo salvo em: /home/anna/tcc/Mineração de Dados - Avaliação e Apresentação/Análise Populacional/hospitais_por_tipo_taxa_por_100_mil.csv
[LOG] ===== FIM: HOSPITAIS POR TIPO — 43 linhas =====


## Conferência dos Resultados

In [7]:
for nome_da_variavel in dicionario_resultados_por_variavel:
    tabela_do_resultado = dicionario_resultados_por_variavel[nome_da_variavel]
    print(nome_da_variavel, "->", tabela_do_resultado.shape[0], "linhas")

dicionario_resultados_por_variavel["Casos"].head()

Casos -> 33267 linhas
Mortalidade -> 38686 linhas
Mamografia -> 11076 linhas
Hospitais -> 26 linhas
Casos por Escolaridade -> 853 linhas
Casos por Raça e Cor -> 853 linhas
Casos por Faixa Etária -> 853 linhas
Hospitais por Tipo -> 43 linhas


,MUNICÍPIOS,CÓDIGO IBGE,ANO,QUANTIDADE,POPULACAO,TAXA_POR_100_MIL
0,abadia dos dourados,3100104,1985,7.0,3609.0,193.959546
1,abaete,3100203,1985,7.0,10210.0,68.560235
2,abre campo,3100302,1985,7.0,9264.0,75.561313
3,acaiaca,3100401,1985,7.0,1874.0,373.532551
4,acucena,3100500,1985,7.0,13254.0,52.814245


In [ ]:

dicionario_resultados_por_variavel["Hospitais por Tipo"]["TIPO"].value_counts()

TIPO
ONCOLOGIA                25
RECONSTRUÇÃO MAMÁRIA     17
TRATAMENTO SINCRÔNICO     1
Name: count, dtype: int64